# Đánh giá toàn bộ quy trình: sinh ứng viên và chọn lại câu

Chạy đúng quy trình suy luận của mô hình đề xuất trên một tập Karpathy (test hoặc val) và đánh giá ở các điểm
được báo cáo trong bài:

| Điểm | Cách chọn câu |
|---|---|
| `before_rerank` | ứng viên đầu tiên (beam search thường), chưa chọn lại |
| `three_score` | ba tín hiệu kế thừa: 0,2·S_base + 0,4·S_align + 0,4·S_belief |
| `plus_s_retr` | ba tín hiệu + 0,1·S_retr |
| `plus_s_obj` | ba tín hiệu + 1·S_obj (ngưỡng r = 3) |
| `five_score` | **năm tín hiệu**, S_obj với r = 3 (cấu hình chính của bài) |
| `five_score_1of5` | năm tín hiệu, S_obj với r = 1 (để so sánh ngưỡng) |

Các bước cho mỗi ảnh:

1. **Bằng chứng** E(I) = YOLO-World (ngưỡng 0,25) ∪ vật thể được ≥ r trong K = 5 câu truy hồi nhắc tới. Câu truy hồi
   lấy từ kho câu mô tả của tập huấn luyện theo cosine CLIP ViT-B/32.
2. **Luật chặn cố định** khi giải mã: trừ δ vào logits của từ vật thể COCO không thuộc E(I). `DELTA = 0` là không
   chặn.
3. **Sinh 5 ứng viên**: 1 câu beam search thường (beam 3), cộng diverse beam search (10 beam, 5 nhóm, phạt đa dạng
   0,8).
4. **Ba tín hiệu kế thừa** (bộ chọn lại câu dạng công thức): S_base là log-xác suất trung bình, S_align là CLIPScore
   (CLIP ViT-B/16), S_belief là belief revision theo vật thể YOLO.
5. **Hai tín hiệu đề xuất:**
   - S_obj = −(số vật thể câu nhắc tới nhưng không thuộc E(I)).
   - S_retr = cosine trung bình giữa câu và K câu truy hồi.
6. **Tổng hợp:** mỗi tín hiệu chuẩn hóa min-max trong năm ứng viên của ảnh, cộng có trọng số, chọn câu có điểm
   cao nhất.

**Cấu hình** (cell CẤU HÌNH): `CHECKPOINT`, `VOCAB_FILE`, `SPLIT`, `DELTA`; trọng số `W_BASE/W_ALIGN/W_BELIEF`,
`W_RETR`, `W_OBJ`; `COMPUTE_SPICE` (cần Java 8, chậm). Mặc định là cấu hình của bài.

**Độ đo:** BLEU-1..4, METEOR, ROUGE-L, CIDEr (tách từ kiểu `simple`), SPICE tùy chọn, CHAIRs, CHAIRi. Nhãn đúng
của CHAIR là vật thể trong COCO instances hợp với vật thể nhắc tới trong 5 câu tham chiếu.

**Lưu kết quả:** file JSON có `metrics` cho từng điểm và `rows` theo định dạng report của repo (ứng viên, xếp hạng,
vật thể YOLO). Vì vậy các script trong `research/scripts/` dùng lại được file này. Vòng sinh câu lưu giữa chừng và
chạy tiếp được nếu bị ngắt.

**Môi trường:** notebook gọi thẳng code của repo `demo/` (không chép lại), nên số liệu khớp với
`retrieval_logit_bias_experiment.py`, `logit_bias_final_eval.py`, `evgate/signal_table.py` và
`sobj_threshold_eval.py`. Cần `torch`, `transformers`, `ultralytics`, `pycocoevalcap`, `nltk`, Java cho METEOR.

Notebook này được sinh bởi `research/scripts/build_eval_notebooks.py`; sửa ở script đó rồi chạy lại để giữ hai
notebook đồng bộ.


In [ ]:
# ==========================================
# CẤU HÌNH
# ==========================================
import json, os, sys, time
from collections import Counter
from pathlib import Path

# Thư mục repo demo (chứa local_captioner.py, chair_metric.py, ...). Trên Kaggle: gắn code repo làm dataset rồi trỏ vào đó.
DEMO_DIR = Path(r"D:/Documents/BaiTap/KiCuiCung/KLCN/demo")
DATASET_DIR = DEMO_DIR.parent / "dataset"
KARPATHY_JSON = DATASET_DIR / "dataset_coco.json"
INSTANCES_JSON = DATASET_DIR / "instances_val2014.json"
IMAGE_DIR = DATASET_DIR / "val2014" / "val2014"      # ảnh val và test của Karpathy đều thuộc COCO val2014
RETRIEVAL_CACHE = DATASET_DIR / "retrieval_cache"    # kho truy hồi đã mã hóa; chưa có thì notebook tự tạo

# Mô hình cần đánh giá
CHECKPOINT = DEMO_DIR / "restval_dynlogits_contrastive_attnpool_epoch40_model.pth"
VOCAB_FILE = DATASET_DIR / "coco2014_karpathy_restval_dynlogits_contrastive_attnpool_vocab.json"

SPLIT = "test"        # "test" | "val"
START, MAX_IMAGES = 0, None   # ảnh xếp theo tên file như các report của bài; vd MAX_IMAGES = 1000 cho tập con val
DELTA = 3.0           # mức chặn của luật cố định; 0 = không chặn (mô hình cơ sở)

YOLO_MODEL = DEMO_DIR / "yolov8l-worldv2.pt"         # YOLO-World
YOLO_CONF, YOLO_MAX_OBJECTS = 0.25, 20
K_RETR = 5            # số câu truy hồi
MAJ = 3               # ngưỡng đa số r: vật thể được >= r câu truy hồi nhắc tới thì vào tập bằng chứng
MAX_NEW_TOKENS = 40
FORCE_CPU = False     # True: không dùng GPU

# Bộ chọn lại câu (giá trị đã khóa trên tập kiểm định, mục 4.2 của bài)
W_BASE, W_ALIGN, W_BELIEF = 0.2, 0.4, 0.4
W_RETR, W_OBJ = 0.1, 1.0
CLIP_SCORER = "openai/clip-vit-base-patch16"
COMPUTE_SPICE = False      # True: thêm SPICE cho before_rerank, three_score, five_score (Java 8, khoảng 1 giờ/5.000 ảnh)
SAVE_EVERY = 100           # lưu giữa chừng sau mỗi chừng này ảnh; chạy lại cell sinh câu sẽ đi tiếp từ chỗ dừng

OUTPUT = DEMO_DIR / "results" / f"eval_rerank_{CHECKPOINT.stem}_d{DELTA:g}_{SPLIT}.json"

In [ ]:
# ==========================================
# NẠP CODE REPO, DỮ LIỆU VÀ NHÃN CHAIR
# ==========================================
if FORCE_CPU:
    import torch
    torch.cuda.is_available = lambda: False   # các module của repo tự chọn thiết bị bằng hàm này
sys.path.insert(0, str(DEMO_DIR))
sys.path.insert(0, str(DEMO_DIR / "research" / "scripts"))
os.environ.setdefault("HF_HUB_OFFLINE", "0")

import numpy as np
import torch
from tqdm.auto import tqdm

import chair_metric
import local_captioner
import vocab as vocab_mod
from evaluate_reranked_quality import compute_metrics
from sretr_experiment import Clip, minmax
from yolo_objects import YoloObjectExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
data = json.loads(KARPATHY_JSON.read_text(encoding="utf-8"))
items = sorted(({"image": im["filename"], "references": [s["raw"] for s in im["sentences"]]}
                for im in data["images"] if im["split"] == SPLIT), key=lambda r: r["image"])
items = items[START: START + MAX_IMAGES] if MAX_IMAGES else items[START:]
paths = [IMAGE_DIR / r["image"] for r in items]
missing = [p.name for p in paths if not p.is_file()]
assert not missing, f"Thiếu {len(missing)} ảnh, vd {missing[:3]}"

OBJECTS, INV = chair_metric.load_synonyms()
fn2id = chair_metric.load_filename_to_cocoid(KARPATHY_JSON)
inst = chair_metric.load_instance_objects_by_cocoid(INSTANCES_JSON, INV)
# Nhãn đúng của CHAIR = vật thể trong COCO instances  U  vật thể nhắc tới trong 5 câu tham chiếu
GT = {r["image"]: chair_metric.ground_truth_from_captions(r["references"], OBJECTS, INV) | inst.get(fn2id.get(r["image"]), set())
      for r in items}


def objs_of(text):
    """Danh sách vật thể COCO (đã quy về tên lớp) mà một câu nhắc tới, như CHAIR."""
    return chair_metric.caption_to_words(text, OBJECTS, INV)[1]


def score(captions, meteor=True, spice=False):
    """BLEU-1..4, METEOR, ROUGE-L, CIDEr (×100), SPICE tùy chọn, CHAIRs/CHAIRi; tách từ kiểu 'simple' như bài."""
    refs = [r["references"] for r in items]
    m = compute_metrics(captions, refs, use_meteor=meteor, use_cider=True, use_spice=spice, tokenizer="simple")
    ch = chair_metric.compute_chair({r["image"]: c for r, c in zip(items, captions)}, GT, OBJECTS, INV)
    out = {k: round(float(v) * (100 if k == "CIDEr" else 1), 2) for k, v in m.items()}
    out.update(CHAIRs=round(ch["CHAIRs"], 2), CHAIRi=round(ch["CHAIRi"], 2),
               mean_len=round(float(np.mean([len(c.split()) for c in captions])), 2))
    return out


print(f"{len(items)} ảnh {SPLIT} | thiết bị {DEVICE} | checkpoint {CHECKPOINT.name} | δ = {DELTA:g}")

In [ ]:
# ==========================================
# BẰNG CHỨNG VẬT THỂ E(I) = YOLO-World  U  {vật thể được >= MAJ trong K câu truy hồi nhắc tới}
# ==========================================
# Kho truy hồi: toàn bộ câu mô tả của tập huấn luyện (train + restval), mã hóa bằng CLIP ViT-B/32, lưu cache.
clip32 = Clip()
caps_f, emb_f = RETRIEVAL_CACHE / "train_captions.json", RETRIEVAL_CACHE / "train_caption_emb_b32.npy"
if caps_f.is_file() and emb_f.is_file():
    BANK_CAPS, BANK_EMB = json.loads(caps_f.read_text(encoding="utf-8")), np.load(emb_f)
else:
    BANK_CAPS = [s["raw"].strip().lower() for im in data["images"] if im["split"] in ("train", "restval") for s in im["sentences"]]
    print(f"Mã hóa kho truy hồi: {len(BANK_CAPS)} câu (chỉ làm một lần)")
    BANK_EMB = clip32.text(BANK_CAPS)
    RETRIEVAL_CACHE.mkdir(parents=True, exist_ok=True)
    caps_f.write_text(json.dumps(BANK_CAPS), encoding="utf-8")
    np.save(emb_f, BANK_EMB)

# K câu gần ảnh nhất theo cosine ảnh-câu (CLIP B/32); kho không chứa câu của tập val/test nên không rò nhãn.
img_emb = clip32.images(paths)
dt = torch.float16 if clip32.device == "cuda" else torch.float32
bank_t = torch.from_numpy(BANK_EMB).to(clip32.device, dt)
NB = np.concatenate([(torch.from_numpy(img_emb[i: i + 256]).to(clip32.device, dt) @ bank_t.T).topk(K_RETR, dim=1).indices.cpu().numpy()
                     for i in range(0, len(img_emb), 256)])
del bank_t

extractor = YoloObjectExtractor(str(YOLO_MODEL), YOLO_CONF, YOLO_MAX_OBJECTS)
vocab_mod.set_object_noun_vocab(extractor.class_names())   # danh từ vật thể cho bằng chứng âm của S_belief
YOLO_OBJS = [extractor.extract(p) for p in tqdm(paths, desc="YOLO-World")]
YOLO_SET = [{INV[o.label] for o in objs if o.confidence >= YOLO_CONF and o.label in INV} for objs in YOLO_OBJS]
NB_COUNTS = [Counter(o for j in idx for o in set(objs_of(BANK_CAPS[j]))) for idx in NB]   # số câu truy hồi nhắc tới mỗi vật thể
EVIDENCE = [y | {o for o, n in c.items() if n >= MAJ} for y, c in zip(YOLO_SET, NB_COUNTS)]
print(f"Trung bình {np.mean([len(e) for e in EVIDENCE]):.2f} vật thể trong E(I) mỗi ảnh")

In [ ]:
# ==========================================
# LUẬT CHẶN CỐ ĐỊNH TRÊN LOGITS:  z'(w) = z(w) - δ · 1[w là từ vật thể COCO, o(w) không thuộc E(I)]
# ==========================================
# Áp ở mọi bước giải mã của cả beam search thường và diverse beam search; từ không phải vật thể giữ nguyên.
IRREGULAR = {"men": "man", "women": "woman", "kids": "kid", "guys": "guy", "mice": "mouse"}
VOCAB = json.loads(VOCAB_FILE.read_text(encoding="utf-8"))
TOK2OBJ = {}
for word, idx in VOCAB.items():
    w = IRREGULAR.get(word) or chair_metric._LEMMATIZER.lemmatize(word, pos="n")
    if w in OBJECTS:
        TOK2OBJ[idx] = INV[w]
print(f"{len(TOK2OBJ)} token trong từ điển ứng với vật thể COCO")

CURRENT_BIAS = None
if not hasattr(local_captioner, "_ORIG_DECODE_STEP"):
    local_captioner._ORIG_DECODE_STEP = local_captioner.ImageCaptioningTransformer.decode_step


def _biased_decode_step(self, memory, sequence):
    logits = local_captioner._ORIG_DECODE_STEP(self, memory, sequence)
    return logits if CURRENT_BIAS is None else logits + CURRENT_BIAS


local_captioner.ImageCaptioningTransformer.decode_step = _biased_decode_step


def rule_bias(evidence):
    if not DELTA:
        return None
    bias = torch.zeros(len(VOCAB))
    for tid, obj in TOK2OBJ.items():
        if obj not in evidence:
            bias[tid] = -DELTA
    return bias.to(DEVICE)

In [ ]:
# ==========================================
# SINH 5 ỨNG VIÊN (có luật chặn) VÀ CHẤM BA TÍN HIỆU
# ==========================================
from clipscore import ClipScorer
from formula_reranker import FormulaReranker
from generate import CandidateCaptionGenerator

generator = CandidateCaptionGenerator(checkpoint_path=CHECKPOINT, vocab_path=VOCAB_FILE, max_candidates=5, num_beams=10,
                                      num_beam_groups=5, diversity_penalty=0.8, max_new_tokens=MAX_NEW_TOKENS, safe_beam_size=3)
clip16 = ClipScorer(model_name=CLIP_SCORER)
formula = FormulaReranker(alpha=W_BASE, beta=W_ALIGN, gamma=W_BELIEF, formula_mode="linear", clip_scorer=clip16,
                          belief_neg_confidence=0.0)

RUN_KEY = {"checkpoint": str(CHECKPOINT), "split": SPLIT, "start": START, "images": len(items), "delta": DELTA,
           "maj": MAJ, "k_retr": K_RETR, "yolo_model": YOLO_MODEL.name, "weights": [W_BASE, W_ALIGN, W_BELIEF]}
ROWS = []
if OUTPUT.is_file():
    old = json.loads(OUTPUT.read_text(encoding="utf-8"))
    if old.get("run") == RUN_KEY:
        ROWS = old["rows"]
        print(f"Chạy tiếp từ ảnh {len(ROWS)}")


def save(metrics=None):
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT.write_text(json.dumps({"run": RUN_KEY, "metrics": metrics, "rows": ROWS}, ensure_ascii=False), encoding="utf-8")


t0, done0 = time.time(), len(ROWS)
for i in tqdm(range(done0, len(items)), desc="Sinh và chấm ứng viên"):
    r, path, objs = items[i], paths[i], YOLO_OBJS[i]
    CURRENT_BIAS = rule_bias(EVIDENCE[i])
    cands = generator.generate(path, objs)
    CURRENT_BIAS = None
    scored = clip16.score(path, cands)
    ranked = formula.rerank(scored, objs)
    ROWS.append({
        "image": r["image"], "references": r["references"],
        "baseline_candidate": cands[0].text, "caption": ranked[0].caption.text,
        "objects": [{"label": o.label, "confidence": o.confidence, "box_xyxy": list(o.box_xyxy) if o.box_xyxy else None} for o in objs],
        "candidates": [c.__dict__ for c in scored],
        "ranking": [{"caption": x.caption.text, "rank": x.rank, "rerank_score": x.rerank_score, "reason": x.reason} for x in ranked],
        "neighbours": [int(j) for j in NB[i]], "evidence": sorted(EVIDENCE[i]),
    })
    if (i + 1) % SAVE_EVERY == 0:
        save()
CURRENT_BIAS = None
save()
if len(ROWS) > done0:
    print(f"{(time.time() - t0) / (len(ROWS) - done0):.2f} giây mỗi ảnh")

In [ ]:
# ==========================================
# HAI TÍN HIỆU ĐỀ XUẤT VÀ CÁC CÁCH CHỌN CÂU
# ==========================================
# S_retr(c) = cosine trung bình giữa câu c và K câu truy hồi (CLIP ViT-B/32)
# S_obj(c)  = -(số vật thể COCO c nhắc tới nhưng không thuộc E(I)); E(I) dùng ngưỡng r (MAJ hoặc 1)
texts = sorted({c["text"] for row in ROWS for c in row["candidates"]})
TEXT_EMB = dict(zip(texts, clip32.text(texts)))

PER_IMAGE = []
for row, idx, ysup, cnt in zip(ROWS, NB, YOLO_SET, NB_COUNTS):
    cands = [c["text"] for c in row["candidates"]]
    rs = {x["caption"]: x["rerank_score"] for x in row["ranking"]}
    base = np.array([rs[t] for t in cands], dtype=np.float64)          # ba tín hiệu (đã chuẩn hóa trong ảnh)
    ce = np.stack([TEXT_EMB[t] for t in cands]).astype(np.float32)
    s_retr = (ce @ BANK_EMB[idx].astype(np.float32).T).mean(axis=1)
    s_obj = {}
    for thr in sorted({MAJ, 1}):
        sup = ysup | {o for o, n in cnt.items() if n >= thr}
        s_obj[thr] = np.array([-sum(o not in sup for o in objs_of(t)) for t in cands], dtype=np.float64)
    PER_IMAGE.append({"cands": cands, "base": base, "s_retr": s_retr, "s_obj": s_obj})


def pick(fn):
    return [p["cands"][int(np.argmax(fn(p)))] for p in PER_IMAGE]


SELECTIONS = {
    "before_rerank": [row["baseline_candidate"] for row in ROWS],
    "three_score": pick(lambda p: p["base"]),
    "plus_s_retr": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"])),
    "plus_s_obj": pick(lambda p: p["base"] + W_OBJ * minmax(p["s_obj"][MAJ])),
    "five_score": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"]) + W_OBJ * minmax(p["s_obj"][MAJ])),
    "five_score_1of5": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"]) + W_OBJ * minmax(p["s_obj"][1])),
}
SOBJ_EQUAL = round(100 * float(np.mean([p["s_obj"][MAJ].max() == p["s_obj"][MAJ].min() for p in PER_IMAGE])), 1)
print(f"S_obj bằng nhau ở cả năm ứng viên (không đổi thứ hạng) ở {SOBJ_EQUAL}% số ảnh")

In [ ]:
# ==========================================
# TÍNH ĐIỂM VÀ LƯU
# ==========================================
SPICE_ON = {"before_rerank", "three_score", "five_score"}
METRICS = {}
for name, caps in SELECTIONS.items():
    METRICS[name] = score(caps, spice=COMPUTE_SPICE and name in SPICE_ON)
    print(f"{name:16s}", {k: METRICS[name][k] for k in ("BLEU-4", "METEOR", "ROUGE-L", "CIDEr", "CHAIRs", "CHAIRi")})
METRICS["sobj_equal_pct"] = SOBJ_EQUAL
for row, cap in zip(ROWS, SELECTIONS["five_score"]):
    row["caption_five_score"] = cap
save(METRICS)
print(f"Đã lưu {OUTPUT}")